In [1]:
import random
import pandas as pd

In [2]:
# We define here a source of damage as an occurrence
# To differentiate attack damage from effect damage, we add the letter A to attack damage
def parse_occurrence(occ):
    if isinstance(occ, str) and occ.upper().endswith('A'):
        return int(occ[:-1]), True
    return int(occ), False

In [3]:
def simulate_process(main_deck, climax, occurrences_parsed, trigger_soul, trigger_blank):
    # Main deck definition
    deck = ['Climax'] * climax + ['Non-Climax'] * (main_deck - climax)
    random.shuffle(deck)
    discard = []
 
    # Trigger deck definition
    trigger_deck = ['Soul trigger'] * trigger_soul + ['No soul trigger'] * trigger_blank
    random.shuffle(trigger_deck)
    trigger_discard = []
 
    # Clock zone and total count initialisation
    clock_zone = []
    total_kept = 0
 
    for base_k, is_A in occurrences_parsed:
        k = base_k
 
        # trigger check system
        if is_A:
            if not trigger_deck and trigger_discard:
                trigger_deck = trigger_discard[:]
                trigger_discard = []
                random.shuffle(trigger_deck)
            if trigger_deck:
                card = trigger_deck.pop()
                trigger_discard.append(card)
                if card == 'Soul trigger':
                    k += 1
 
        # main damage system
        drawn_this_occurrence = []
        occurrence_failed = False
        halted = False
        draws_done = 0
 
        while draws_done < k:
            if len(deck) == 0:
                # refresh penalty system
                if not discard:
                    halted = True
                    break
                new_deck = discard[:]
                discard = []
                random.shuffle(new_deck)
                clock_card = new_deck.pop()  
                clock_zone.append(clock_card)
                total_kept += 1 
                deck = new_deck
                if len(deck) == 1:
                    halted = True
                    break
                continue
 
            card = deck.pop()
            drawn_this_occurrence.append(card)
            draws_done += 1
            if card == 'Climax':
                occurrence_failed = True
                break
 
        if halted:
            discard.extend(drawn_this_occurrence)
            break
 
        if occurrence_failed:
            discard.extend(drawn_this_occurrence)
        else:
            # damage count upgrade system
            total_kept += len(drawn_this_occurrence)
            clock_zone.extend(drawn_this_occurrence)
 
            # level-up system
            while len(clock_zone) >= 7:
                batch = clock_zone[:7]
                clock_zone = clock_zone[7:]
                white_idx = next((i for i, c in enumerate(batch) if c == 'Non-Climax'), None)
                if white_idx is not None:
                    batch.pop(white_idx)
                else:
                    batch.pop(0)
                discard.extend(batch)
 
    return total_kept

In [4]:
def simulation(N_list, p_list, occurrences, trigger_soul, trigger_blank, n_trials=100000):
    """
    Parameters
    ----------
    N_list : list[int]        main deck size list
    p_list : list[int]        climax list
    occurrences : list        finisher damage string
    trigger_soul : int        cards with soul triggers in trigger deck
    trigger_blank : int       cards without soul trigger in trigger deck
    n_trials : int            Monte Carlo simulations for each [N,p] combination
 
    Result
    ------
    Each line corresponds to a [main_deck, climax] combination
    Each column correspond to the associated probability where at least X damage go through the simulation
    """
    occurrences_parsed = [parse_occurrence(o) for o in occurrences]
    max_possible = sum(k + (1 if is_A else 0) for k, is_A in occurrences_parsed)
 
    rows = []
    for main_deck in N_list:
        for climax in p_list:
            totals = [
                simulate_process(main_deck, climax, occurrences_parsed, trigger_soul, trigger_blank)
                for _ in range(n_trials)
            ]
            row = {'Nb card deck': main_deck, 'Nb climax': climax}
            for x in range(0, max_possible + 1):
                row[f'X_min={x}'] = round(sum(1 for t in totals if t >= x) / n_trials, 4)
            rows.append(row)
 
    return pd.DataFrame(rows)

In [5]:
table = simulation(
        N_list=[20,25,30],
        p_list=[4,6,8],
        occurrences=[4,'3A',4,'3A',4,'3A'],
        trigger_soul = 15,
        trigger_blank = 35,
        n_trials=10000,
        )

table.to_csv('resultats_simulation.csv', index=False)
table

,Nb card deck,Nb climax,X_min=0,X_min=1,X_min=2,X_min=3,X_min=4,X_min=5,X_min=6,X_min=7,...,X_min=15,X_min=16,X_min=17,X_min=18,X_min=19,X_min=20,X_min=21,X_min=22,X_min=23,X_min=24
0,20,4,1.0,1.0000,1.0000,1.0000,1.0000,0.9997,0.9978,0.9351,...,0.0066,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
1,20,6,1.0,0.9387,0.9387,0.9387,0.7417,0.4812,0.4812,0.4063,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
2,20,8,1.0,0.6683,0.6683,0.6683,0.4009,0.1282,0.1282,0.0975,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
3,25,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9757,...,0.1141,0.0311,0.0153,0.0100,0.0018,0.0001,0.0000,0.0000,0.0000,0.0
4,25,6,1.0,0.9871,0.9871,0.9871,0.9066,0.7820,0.7820,0.7100,...,0.0095,0.0010,0.0004,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
5,25,8,1.0,0.8805,0.8805,0.8805,0.6825,0.4185,0.4185,0.3514,...,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
6,30,4,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.9893,...,0.2735,0.1230,0.0897,0.0666,0.0273,0.0061,0.0035,0.0014,0.0001,0.0
7,30,6,1.0,0.9964,0.9964,0.9964,0.9600,0.9026,0.9026,0.8520,...,0.0575,0.0156,0.0082,0.0056,0.0012,0.0002,0.0001,0.0001,0.0000,0.0
8,30,8,1.0,0.9551,0.9551,0.9551,0.8274,0.6510,0.6510,0.5791,...,0.0087,0.0013,0.0006,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
